# Measuring is the hard part

Notes from a few weeks of work on Kaggriculture. This is **not** an agent — there is no
policy in here to copy. It is the evaluation harness, because that is where almost all of
our progress actually came from.

The single most useful thing I can tell you: **every large rating jump we got was a
measurement fix, never a better agent.** We tuned ~90 constants with an evolutionary
league for days and moved very little. Then we found we were optimising against the wrong
engine version, and separately that 45% of our harvest actions were silently doing
nothing, and those two fixes were worth more than the entire preceding tuning effort.

If you are stuck, the odds are good that you are measuring the wrong thing, not choosing
the wrong parameters.

Contents:

1. Rank by win rate against a field, never by margin against one baseline
2. Self-play lies in a shared market
3. Three seed pools, and the failure that made us build the third
4. **[runnable]** The no-op audit: instrument the engine, not the agent
5. The action-budget law
6. **[runnable]** Comparing two agents honestly
7. The replay oracle: proving which engine the competition runs
8. The compute budget nobody is using
9. A list of things that did not work

---
## 1. Rank by win rate against a field, never by margin against one baseline

This one cost us the most time before we understood it, and it is the least intuitive.

The obvious way to evaluate a candidate is to play it against your current best agent and
look at the average score difference. This is **actively misleading** in a two-player
shared-market game. Here are two candidates we generated on the same day:

| candidate | mean margin vs baseline | win rate vs the whole field | head to head |
|---|---|---|---|
| A | **+$41,695** | 57% | loses to B **30-0** |
| B | +$8,265 | **96%** | — |

Candidate A has a *five times larger* margin against the baseline and loses every single
game to the agent we actually shipped. We also had an agent beat the baseline 48 games out
of 50 and still rank below it against the field.

The game is strongly **non-transitive**: A beats B, B beats C, C beats A. Overall win rate
and head-to-head results routinely disagree near the top. So a duel against one opponent
tells you almost nothing, and a *margin* in a duel tells you less than nothing, because
margin is inflated by opponents you crush and says nothing about the ones that beat you.

What we do instead: an accumulating round-robin where every candidate plays every other
candidate and every past champion, ranked by win rate. Wins, not dollars.

**Corollary that bit us:** a champion's *lifetime* win rate is not comparable across
champions, because each accumulated its record against the field that existed at the time.
One of ours read 91% lifetime but 86% over its last 120 games — exactly level with a rival
it appeared to dominate. Compare recent windows, not lifetime totals.

---
## 2. Self-play lies in a shared market

Both farms sell into **one** market, and the price of a good falls as inventory rises. So
if you cut your own production, you relieve saturation *for both copies of yourself* and
**both absolute scores go up**.

We had a configuration that scored the highest absolute self-play score of anything we had
ever tried. It lost 74% of its ladder games. It was not producing more; it was competing
less.

Two rules fall out of this:

- **Never rank a configuration by its absolute self-play score.** Rank by win rate.
- When a change makes your score go up *and your opponent's score go up too*, you have
  probably just produced less. We started treating "opponent's absolute score rose" as a
  warning signature, and it caught a lot of bad ideas. Every single one of our
  production-cutting experiments showed it.

There is a second, subtler version of this. Comparing absolute scores *across different
episodes* is meaningless here, because the random shop draw at the start of a season
explains most of the variance in final score — we fit roughly

```
final_score ~= -51,089 + 185.7 * (demand-days available to you)      r = 0.915
```

and the same player swings between $68k and $154k depending on the draw. Within a single
episode the draw is shared by both players, so the *relative* score is meaningful. That is
the only score comparison worth making.

---
## 3. Three seed pools, and the failure that made us build the third

We keep evaluation seeds in strictly disjoint pools. The important thing is not the
numbers, it is that a pool used for *selection* can never be used for *validation*.

| pool | used for |
|---|---|
| gate | the final pass/fail before we spend a submission |
| holdout | seeds nothing has ever trained or been selected on |
| panel | games against **other people's agents**, not our own |

The first two are standard hygiene. The third exists because of a specific failure that
took us days to diagnose, and it is the part I think generalises best.

**What happened.** We promoted six successive champions in eleven hours. Every one of them
beat the previous champion convincingly and reproducibly. Not one of them beat our
*older* agent on the actual leaderboard. The best agent we had was several generations
stale and we kept replacing it with worse ones.

Our first hypothesis was seed overfitting. We built the virgin holdout pool to test it,
and the improvement **survived completely** — 60% win rate, +$1,151/game, t = +5.28 on
seeds nothing had ever touched. So the seeds were clean and the improvement was real.

Then we downloaded two strong public agents, and the same comparison against *them* read
**+$65/game, t = +0.13**. Nothing. Zero.

The improvements were real and they were **opponent-specific**. Our entire selection stack
— the evolving population, the round-robin, the hall of fame, the gate — contained
nothing but our own descendants. So it could reward "beats its own siblings" indefinitely
without ever producing an agent that beats a stranger, and it did exactly that, six times
in a row, while every statistic we had said we were improving.

**The fix** was to fold foreign agents into the evolutionary fitness function itself (a
fraction of each fitness evaluation is played against a downloaded agent rather than a
sibling), and to require any submission to clear the foreign panel. The first candidate
produced afterwards read +$1,671/game, t = +3.90 against outsiders — the first one in that
entire era to move that number at all.

**The lesson:** if every agent in your evaluation loop descends from a common ancestor,
your loop cannot distinguish "stronger" from "better adapted to my own family". Get at
least one agent into the loop that you did not write. Public notebooks are right there.

> On extracting a public agent to test against: parse the notebook JSON and pull the source
> cells out. **Do not execute a downloaded notebook.** One agent we used publishes a
> sha256 of its own payload, which let us confirm our extraction was byte-exact rather
> than merely plausible — worth checking if the author provides it.

---
## 4. The no-op audit: instrument the engine, not the agent

This is the single highest-value diagnostic we built, and it is about ten lines.

The engine silently ignores illegal actions. It does not raise, it does not warn — your
unit simply spends its turn doing nothing and the game moves on. An agent can be throwing
away a third of its actions and look completely healthy from the outside.

You cannot see this from inside your agent, because your agent believes it issued a valid
action. So **wrap the engine's action handler**, snapshot the game state before and after,
and count the calls that changed nothing.

When we did this, we found **45.5% of our HARVEST actions did nothing at all**. The cause:
the engine refuses to harvest a crop before its `first_yield_day` and returns silently,
and our melons hit their yield cap several days before that day arrived. So for two to
three days per melon tile we walked a unit onto the tile every single turn to accomplish
nothing. One `if` statement fixed it and it was our second-largest gain of the project.

The cell below runs the audit on the two agents that ship with the environment.

In [ ]:
import json
from kaggle_environments import make
from kaggle_environments.envs.kaggriculture import kaggriculture as K


def noop_audit(agents, seed=9001):
    """Count engine actions that changed nothing.

    Wraps _apply_unit_action, snapshots the mutable farm + private state either
    side of the call, and compares. PASS is tracked separately: a deliberate pass
    is a legitimate choice, not waste, and it would otherwise dominate the count.
    """
    original = K._apply_unit_action
    stats = {}

    def hooked(farm, private, idx, action, *args, **kwargs):
        before = (json.dumps(farm, sort_keys=True), json.dumps(private, sort_keys=True))
        result = original(farm, private, idx, action, *args, **kwargs)
        after = (json.dumps(farm, sort_keys=True), json.dumps(private, sort_keys=True))
        name = action[0] if isinstance(action, (list, tuple)) and action else str(action)
        row = stats.setdefault(name, [0, 0])
        row[0] += 1
        row[1] += (before == after)
        return result

    K._apply_unit_action = hooked
    try:
        make("kaggriculture", configuration={"seed": seed}, debug=False).run(agents)
    finally:
        K._apply_unit_action = original          # always restore, even on failure
    return stats


def report(label, stats):
    real = {k: v for k, v in stats.items() if k != "PASS"}
    total = sum(v[0] for v in real.values())
    wasted = sum(v[1] for v in real.values())
    print(f"{label:10s} {total:5d} real actions   {wasted:5d} wasted   "
          f"{100 * wasted / max(1, total):5.1f}%")
    for name, (n, bad) in sorted(real.items(), key=lambda kv: -kv[1][1])[:5]:
        if bad:
            print(f"    {name:10s} {n:5d} issued  {bad:5d} no-op  {100 * bad / n:5.1f}%")


for name in ["starter", "random"]:
    report(name, noop_audit([name, name]))

The `starter` agent wastes nothing — it only ever issues actions it has already verified.
The `random` agent wastes about half of everything it does, and the breakdown localises it
precisely: nearly every HARVEST and WATER it issues is illegal.

That per-action breakdown is the useful part. A single waste percentage tells you that
something is wrong; the breakdown tells you *which rule you have mis-modelled*.

**Two traps that cost us real time here.**

*Count actions, not intentions.* Our first version counted "harvest tasks assigned", which
massively over-counted, because a unit assigned a distant harvest spends most of its turns
walking there and emits MOVE. Hook the engine, not your planner.

*If you audit recorded games, use raw replays with both players' actions.* We first ran
this against replays where the opponent was replaced by a passing agent. The player's money
then diverges from the real game, their scripted purchases fail, their later actions land
on tiles they never bought, and you get a flood of fake violations. Replays that contain
both seats reproduce the original result exactly, and then every no-op you count is real.

We eventually ran this across the top of the leaderboard. Waste tracked score almost
monotonically: the strongest player wasted **10 actions out of 145,194**. Players in the
12-18% range were scoring 30-40% lower. It is a very cheap way to find out whether you have
a bug or a strategy problem — a large gap on a purely *mechanical* statistic is a bug.

---
## 5. The action-budget law

Having built the audit, the obvious next move is to drive waste to zero. **This is a trap**,
and it is worth understanding why, because it generalises to almost every "efficiency" idea
in this game.

Each unit acts once per turn. That is the binding constraint — not money, not land. We
routinely leave 16 unlocked tiles empty because we do not have the actions to farm them.

So any fix that *spends an action* to prevent a loss competes against whatever that action
would otherwise have done, and the displaced action is usually worth more than the loss:

| fix | what it removed | what it cost |
|---|---|---|
| reserve shed stock per unit | **100%** of failed pickups | −$598/game |
| harvest one tick before the animal's cap clips it | the entire measured 13 units/game | −$1,625/game |
| de-duplicate couriers | all duplicate trips | lost 14 of 20 games |

The failed-pickup case is the clearest. Three units stand at the shed, it holds 2 wheat,
the first takes it and the other two waste their turn. Reserving stock fixes that
perfectly — and loses, because the blocked unit wanders off to do something else, so when
wheat arrives next turn nobody is standing there to collect it. **The redundancy was
insurance.** The wasted action was cheaper than not having a unit in position.

We hit this five separate times, including a case where a plainly fatal bug in our own
agent (sowing seeds on the last turn of the day, where they always die overnight) turned
out to be *load-bearing*: removing it made us worse and we still do not fully know why.

**Use the audit to find actions that silently do nothing, then test the fix. Do not use it
to drive a waste counter to zero.** Measurable waste is not the same as recoverable waste.

The profitable direction is the mirror image: find something you are spending actions *on*
that is worth nothing, and stop. We found we were feeding animals on the final day, which
produces goods after the last harvest — pure waste. Removing it gained $864/game, and
**most of the gain was not the recovered goods**, it was the freed actions going into
selling.

---
## 6. Comparing two agents honestly

A small checklist that we arrived at by getting each of these wrong at least once.

- **Play both seats on every seed.** The seats are not symmetric. Two identical agents do
  not draw — they split on seat asymmetry.
- **Use common seeds and pair the results.** Between-episode variance is enormous (the shop
  draw again); paired differencing removes almost all of it.
- **Report a t-statistic, not a mean.** Our bar is t > 2 on the full sample.
- **Watch the margin as the sample grows.** One of ours read +$793 (t = +2.32) at 120 games
  and +$420 (t = +1.70) at 200. That is regression toward no-difference, not a finding. We
  submitted it anyway and it rated ~700 points below the agent it replaced.
- **A margin of exactly $0.00 means you are comparing a file with itself.** This happens far
  more than you would think, usually because a patch script's anchor string did not match
  and it silently rewrote nothing. Assert the anchor exists. Diff the two files you are
  actually about to compare.
- **A perfectly inert result can also mean the code never executed.** We had a feature read
  exactly $0.00 for a different reason: its trigger threshold was never reached in any game.
  That is not a falsification, it is a binding failure. Before tuning any constant, check in
  one instrumented game whether the quantity ever actually reaches it. Three of our
  "surprisingly inert" parameters turned out to be gated by a *different* constant upstream.

In [ ]:
import math
from statistics import mean
from kaggle_environments import make


def duel(agent_a, agent_b, seeds, verbose=True):
    """Paired comparison over common seeds, both seats. Returns (wins, t)."""
    diffs = []
    for seed in seeds:
        for swap in (False, True):
            pair = [agent_b, agent_a] if swap else [agent_a, agent_b]
            env = make("kaggriculture", configuration={"seed": seed}, debug=False)
            env.run(pair)
            r = [s["reward"] or 0 for s in env.steps[-1]]
            a, b = (r[1], r[0]) if swap else (r[0], r[1])
            diffs.append(a - b)

    n = len(diffs)
    m = mean(diffs)
    sd = (sum((d - m) ** 2 for d in diffs) / (n - 1)) ** 0.5 if n > 1 else 0.0
    se = sd / math.sqrt(n) if n else 0.0
    t = m / se if se else 0.0
    wins = sum(d > 0 for d in diffs)
    if verbose:
        print(f"{n} games ({len(seeds)} seeds x 2 seats)")
        print(f"  wins   {wins}/{n}  ({100 * wins / n:.0f}%)")
        print(f"  margin ${m:,.0f}/game   stderr ${se:,.0f}   t = {t:+.2f}")
        print(f"  verdict: {'SHIP' if t > 2 else 'not proven — do not spend a submission'}")
    return wins, t


duel("starter", "random", seeds=range(9001, 9007))

That demo pairs two agents of wildly different strength, so the t-statistic is absurd —
ignore the magnitude. The number to look at is **stderr $6**: paired differencing over
common seeds with both seats played is extraordinarily tight, which is what makes it
possible to resolve a $400 effect in a game where single-episode scores swing by $80,000.
Real comparisons between two serious candidates land at t = 2 to 16.

One more check before shipping anything, which is specific to shared-market games: confirm
the improvement is **production, not denial**. Compare the absolute scores as well as the
margin. If your candidate wins by making the opponent's score collapse while its own score
stays flat, you have found a way to crash the market rather than a way to farm better, and
it will not survive contact with an opponent who does not play into it. We require our own
absolute output to rise.

---
## 7. The replay oracle: proving which engine the competition runs

For over a week, every measurement we made was against the wrong version of the game.

The environment was updated mid-competition. Town-centre demand dropped by about 79%, a
demand ramp was deleted outright, and shops began being drawn **with replacement** (so a
season can genuinely contain three bakeries and no yarn store). Our entire tuned
configuration had been selected against mechanics that no longer existed, and our internal
ranking of our own agents was *inverted* relative to the real leaderboard — the agent our
harness rated worst was our best-performing submission.

You do not have to guess about this, because replays are a **verifiable oracle**. A replay
contains both players' actions and the episode seed. Feed those actions back into a
candidate engine version and the correct one reproduces the recorded final money **exactly,
to the cent**. The wrong one does not come close.

| replays | under the old version | under the new version |
|---|---|---|
| 101 older episodes | **101/101 exact** | 0/101 |
| 55 newer episodes | 0/55 | **55/55 exact** |

That is a completely unambiguous answer, and it is worth re-running whenever the hosts
announce anything. It also verifies the *whole* engine at once, which is a stronger
statement than reading a changelog.

**The trap, and you will hit it.** `kaggle_environments` records an action in the step that
*produced* it, so the action applied at time *t* lives at `steps[t+1]`, not `steps[t]`.
Pairing them naively made 40% of a strong player's actions look illegal — including 100% of
several action types, which is the tell. It also produces final-money errors that look
plausible ($241 off, $4,000 off) rather than obviously broken, which is exactly what makes
it dangerous. Try both offsets and keep the one that reconciles.

Two independent public notebooks confirmed the version change for us after the fact. We
would have saved a week by checking directly.

---
## 8. The compute budget nobody is using

Worth stating plainly, because we ignored it for weeks:

| | |
|---|---|
| our agent's decision time | **0.22 ms** per turn (0.64 ms worst case) |
| whole game | 0.16 s |
| per-step limit | 1,000 ms |
| whole-episode limit | 1,200 s |

That is roughly a **4,500x** headroom. Every result in this notebook was produced by a
reactive policy that decides in a fifth of a millisecond and never looks ahead a single
turn.

We ran a ceiling probe on search: simulate the game forward under a few candidate postures
each day and play the winner. With full information the result was the largest effect we
have ever measured — **+$4,352/game, winning 40 out of 40**.

Then it mostly evaporated. Priced honestly:

| information available to the search | gain | t |
|---|---|---|
| everything (oracle) | +$4,352 | +12.80 |
| minus the episode seed | +$1,297 | +4.44 |
| minus the opponent's private inventory | +$362 | +1.16 |
| **minus both (what you can actually know)** | **+$574** | **+1.33** |
| oracle information, but facing a *stranger* | +$778 to +$994 | < 1.1 |

Two separate losses, and they stack: you do not know what the opponent is holding, and even
with perfect information a search tuned against yourself transfers poorly to an opponent
you have not seen. Not knowing the opponent's pantry costs far more than not knowing the
seed.

Three design constraints we measured, if you try this:

1. **Your rollouts must reach the end of the game.** The gain is monotonic in horizon and
   short horizons are *actively harmful* (−$3,312 at 10 steps). Final money at a near
   horizon is maximised by not investing, so a shallow search confidently sells its stock
   and stops planting. Any evaluation that is not final money will mislead you here.
2. **The rollout policy must resemble your real one.** Substituting a simple built-in agent
   inside the rollouts flipped the result negative — a weak stand-in ranks the options
   wrongly. This kills the obvious speedup.
3. **The cost is your policy, not the engine.** The engine is 0.025 ms/step, about 5% of
   rollout cost; your own agent, invoked twice per simulated step, is the other 95%.

I am recording this as *promising but unproven*. The headroom is real and almost nobody is
using it, but the honest version of the edge does not yet clear t > 2 for us.

---
## 9. Things that did not work

Offered to save you the time. All measured over 20-240 games, both seats, against our own
best agent of the day. This is a partial list.

**Market behaviour.** Holding stock back for a better price loses monotonically, at every
threshold we tried, and gets worse the harder you hold. Sell-price floors: inert or
negative. Capping order sizes to avoid moving the price: −$60 at a 12-unit cap, −$11,302 at
a cap of 3. Re-ranking sells by urgency instead of revenue: −$2,783.

The reason is that selling is **throughput-limited, not price-limited**. Only 10 market
orders per turn are processed, produce keeps arriving, and the shed discards anything over
its cap at end of day. Anything that sells less often or less per order simply bins stock.
Check whether you are actually in a glut before optimising for one — we were not, and the
price we realised was well *above* base for our main crops.

**Reading the opponent.** Seven attempts, seven losses. Their whole farm is public — tiles,
crops, animals, money — so this looks like free signal. Selling earlier when they are about
to dump the same product: −$334. Tilting our crop mix away from theirs: −$2,215. Buying
feed earlier when their herd is large: −$4,597. Matching a product they are flooding rather
than ceding it: −$3,891.

The opponent's absolute score rose in *every one*. In this shared market, any deviation
from your own best independent play seems to help them more than you. Note also that
ceding a flooded product is clearly wrong — competing head-on against an opponent who went
all-in on milk won us 20 games out of 20 — so the failure is specifically in *reacting*,
not in competing.

**Imitation and RL.** Behaviour cloning from 168k top-player decisions plateaued far below
our hand-written agent, and **9x more data made it worse**. The reason is locatable:
movement is 52% of all decisions, and the correct move depends on a target chosen several
turns earlier, so identical board states require different moves. A reactive policy
literally cannot represent the difference. Better architectures raised move accuracy a lot
and playing strength barely at all. PPO over a learned decision layer oscillated around zero
for 139 iterations.

**Copying the top players' statistics.** They field far more units than us on day 0. Forcing
ourselves to match cost between $2,841 and $9,211/game, losing essentially every game. Their
unit count is load-bearing for a *hardcoded route* — which never has to decide anything and
can profitably staff a bigger farm — and actively harmful for a reactive assignment policy,
which gets worse as units outnumber worthwhile tasks. Their composition did not transfer
either.

Which leads to the last one. We recorded 20 strong routes that reproduced exactly, and
replayed them elsewhere:

| condition | route scores | as % of its recorded result |
|---|---|---|
| its own seed, opponent passing | $132,779 | **140.6%** |
| its own seed, against a real opponent | $66,666 | 68.2% |
| a **foreign** seed, against a real opponent | $68,918 | 73.8% |

Routes are **seed-robust and opponent-fragile**. Moving a route to a seed it has never seen
barely hurts it; putting a competitor in front of it halves it. That makes sense — a route
sells on pre-planned turns, and its recorded revenue assumed one specific opponent's selling
pattern. A top player reported the same from the other side: freezing a rival's observed
trace scored 0 for 50.

Route replay clearly *can* win — the top of the leaderboard does it. But a recorded route is
behaviour under one state path, not a portable policy, so it has to be fitted fresh against
a panel of opponents with closed-loop control on top. It is a from-scratch build, not a
transplant.

---
## Closing

If I could give my past self one instruction it would be: **spend the first week building
the harness, not the agent.**

Concretely, the four things that paid:

1. Rank by win rate against a **field**, never margin against one baseline.
2. Get at least one agent you did not write into your evaluation loop, early. Otherwise you
   cannot tell "stronger" from "better at beating my own relatives", and you will not find
   out until the leaderboard tells you, days later.
3. Verify which engine you are actually playing, with replays, not with a changelog.
4. Instrument the engine to find actions that silently do nothing — then **test the fix**
   rather than assuming a measured waste is a recoverable one.

And the meta-point behind all four: when a search looks converged, check what you are
measuring before you tune harder. Ours was converged twice, and both times the problem was
the instrument.

Questions and corrections welcome — several numbers here contradict things I believed
confidently a week ago, so I would not be surprised to be wrong about more.